# 04. Multi-Query Retrieval 
— 하나의 질문을 여러 검색 관점으로 확장하기

## 이번 실습의 위치

앞 실습에서는 **하나의 사용자 질문을 하나의 더 좋은 검색 질문으로 바꾸는 Query Rewriting**을 학습했다.

```text
01 Naive RAG
        ↓
02 Document Processing
   Docling + Structure-aware Chunking
        ↓
03 Query Rewriting
   1 Question → 1 Better Query
        ↓
04 Multi-Query Retrieval
   1 Question → Multiple Queries
```

이번 실습에서는 **하나의 질문을 의미가 다른 여러 검색 표현으로 확장**하여
단일 질의에서 놓칠 수 있는 관련 문서를 더 넓게 찾는 방법을 학습한다.

## 학습 목표

실습 후 다음 내용을 설명할 수 있어야 한다.

1. Multi-Query Retrieval이 필요한 이유를 설명한다.
2. Query Rewriting과 Multi-Query Retrieval의 차이를 설명한다.
3. 하나의 질문에서 여러 검색 질의를 생성한다.
4. 각 질의별 검색 결과를 합치고 중복 문서를 제거한다.
5. Single Query와 Multi Query의 검색 범위를 비교한다.
6. Multi Query가 **Recall을 높이기 위한 전략**이라는 점을 이해한다.
7. 검색 문서가 많아지는 것이 곧 성능 향상을 의미하지는 않는다는 점을 이해한다.

> 이번 실습에서는 **Multi Query 자체에 집중한다.**  
> Dense + BM25 + Hybrid + RRF는 다음 `05_retrieval_engineering_ai_trend.ipynb`에서 별도로 다룬다.


# 1. 02번에서 만든 Vector DB 불러오기

이번 실습에서도 동일한 문서와 동일한 Dense Retriever를 사용한다.

```text
chroma_db/ai_trend_report_docling
```

즉, 이번 단계에서 바뀌는 것은 **검색 질의의 개수**뿐이다.

```text
Single Query
1개 질문 → Dense Retrieval

Multi Query
1개 질문 → 여러 검색 질문 → 각각 Dense Retrieval
```

이렇게 해야 검색 결과 변화가 Multi Query 때문인지 비교하기 쉽다.


In [1]:
import sys
sys.path.append("..")
from common_config import embedding_model, llm_connect  # 공통 설정: 임베딩 모델 / LLM 연결 함수

from pathlib import Path
from langchain_chroma import Chroma

# 02번 실습에서 만든 Chroma DB 경로와 컬렉션 이름
DB_PATH = Path("chroma_db/ai_trend_report_docling")
COLLECTION_NAME = "ai_trend_report_2026_docling"

# DB가 없으면 02번을 먼저 실행해야 하므로 명확한 에러를 발생시킨다
if not DB_PATH.exists():
    raise FileNotFoundError(
        "02_document_processing_ai_trend.ipynb를 먼저 실행하여 "
        f"{DB_PATH}를 생성하세요."
    )

embeddings = embedding_model()
# temperature=0: 질의 생성이 매번 같은 결과를 내도록 고정 (재현성)
llm = llm_connect(temperature=0, max_tokens=512)

# 기존에 저장된 벡터 DB를 다시 연결 (새로 임베딩하지 않음)
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH),
)

print(f"불러온 chunk 수: {vector_store._collection.count()}")


불러온 chunk 수: 48


# 2. 먼저 Single Query 검색을 확인한다

Multi Query를 바로 적용하지 않고, 먼저 **질문 하나로 검색한 결과**를 확인한다.

예제 질문:

```text
2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
```

이 질문 하나만 embedding하여 검색하면,
표현이 비슷한 문서는 잘 찾을 수 있지만 관련 내용을 다른 표현으로 설명한 chunk는 놓칠 수 있다.


In [2]:
def inspect_results(query: str, k: int = 4):
    """Dense Retrieval 결과를 metadata와 함께 출력한다."""
    # 질문을 embedding하여 가장 가까운 k개 chunk와 거리(distance)를 가져온다
    results = vector_store.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"[검색 질문] {query}")
    print("cosine distance: 낮을수록 더 유사함")
    print("=" * 100)

    for rank, (doc, score) in enumerate(results, start=1):
        meta = doc.metadata  # chunk_id, pages, heading 등 02번에서 저장한 메타데이터
        print(
            f"\n[{rank}] distance={score:.4f}"
            f" | chunk_id={meta.get('chunk_id')}"
            f" | pages={meta.get('pages')}"
        )
        print(f"heading={meta.get('heading')}")
        print("-" * 100)
        # 본문은 앞부분 550자만 미리보기로 출력
        print(doc.page_content[:550].replace("\n", " "))

    return results


In [3]:
question = "2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?"

# 기준선(baseline): 질문 1개로 검색한 결과 (Single Query)
single_results = inspect_results(question, k=4)


[검색 질문] 2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
cosine distance: 낮을수록 더 유사함

[1] distance=0.3202 | chunk_id=10 | pages=5
heading=AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
----------------------------------------------------------------------------------------------------
AI@Data Report ■ 2026년 AI 생태계 방향성 전망 - 산업·기술·정책이 서로 다른 변화 축을 보이지만, 2026년에는 세 변화가 동시에 심화되며 AI 생태계의 구조 재편이 가속화될 가능성이 있음 - Ÿ 산업은 도입 확대, 기술은 기능 고도화, 정책은 책임·안전 중심의 제도화가 각각 진전될 것으로 예상되며, 이러한 흐름은 AI 활용의 범위, 속도, 규율 체계에 모두 영향을 미칠 것으로 전망됨

[2] distance=0.3710 | chunk_id=38 | pages=17
heading=4.2 2026년 AI 산업 전망
----------------------------------------------------------------------------------------------------
4.2 2026년 AI 산업 전망 - 2026년 AI 시장은 선도 기업을 중심으로 내재화가 빠르게 진행되며 시장 규모가 지속 확대될 전망 - Ÿ AI 활용이 여러 산업군에서 실험 단계를 넘어 일부 핵심 공정과 서비스 운영에 본격 도입되기 시작하면서 AI가 기업 운영 인프라로 자리 잡아가는 흐름이 확산될 가능성이 높음 - Ÿ 특히 대기업·디지털 강도가 높은 산업을 중심으로 의사결정·운영·고객 대응 등 주요 영역에 AI가 단계적으로 통합되며 AI 관련 시장 수요가 꾸준한 성장세를 이어갈 것으로 평가됨 - AI 활용이 확산되면서 고품질·도메인 특화 데이터(버티컬 데이터)의 중요성이 크게 - 높

## Single Query의 한계

하나의 질문은 하나의 embedding vector로 표현된다.

```text
사용자 질문
     ↓
Embedding
     ↓
하나의 Query Vector
     ↓
Vector Search
```

하지만 같은 의도도 다양한 표현으로 나타날 수 있다.

예:

```text
2026년 AI 기술 발전 방향은?
AI 기술의 주요 고도화 흐름은?
AI 기술의 기능과 적용 환경은 어떻게 변화하는가?
AI 기술에서 중요해지는 기술적 변화는?
```

의미는 유사하지만 각각 다른 문서 표현과 더 잘 맞을 수 있다.

따라서 **한 질문을 여러 검색 표현으로 확장하여 검색 범위를 넓히는 것**이 Multi-Query Retrieval이다.


# 3. Query Rewriting과 Multi Query의 차이

둘은 비슷해 보이지만 목적이 다르다.

| 구분 | Query Rewriting | Multi Query |
|---|---|---|
| 입력 | 질문 1개 | 질문 1개 |
| 출력 | 더 좋은 질문 1개 | 의미가 다른 질문 여러 개 |
| 주요 목적 | 질문 명확화 | 검색 범위 확대 |
| 검색 횟수 | 1회 | 여러 회 |
| 기대 효과 | 검색 표현 개선 | Recall 개선 |

```text
Query Rewriting
Q → Q'

Multi Query
Q → Q1
  → Q2
  → Q3
```

### 핵심

Multi Query의 목적은 **질문을 여러 주제로 바꾸는 것**이 아니다.

원래 질문의 의미를 유지하면서,
**같은 정보 요구를 서로 다른 표현으로 검색하는 것**이다.


# 4. LLM으로 여러 검색 질문 만들기

처음 Advanced RAG를 배우는 단계에서는 내장 Retriever를 바로 사용하는 것보다,
**질의 생성 → 검색 → 병합** 과정을 직접 구현하는 것이 이해하기 쉽다.

이번 실습에서는 원본 질문에서 **3개의 검색 질의**를 생성한다.

좋은 Multi Query:

```text
원본:
2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?

Q1. 2026년 AI 기술의 주요 발전 방향은 무엇인가?
Q2. AI 기술의 기능 고도화와 적용 환경은 어떻게 변화할 것으로 예상되는가?
Q3. 2026년 AI 기술 경쟁에서 중요해지는 변화 요인은 무엇인가?
```

나쁜 Multi Query:

```text
Q1. AI 기술 전망은?
Q2. AI 시장 규모는?
Q3. AI 규제 벌금은?
```

Q2, Q3는 원래 질문에서 벗어난 새로운 주제이다.


In [4]:
import re

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Multi-Query 생성 프롬프트
# 핵심 제약: 원본 의도 유지 / 새 주제 추가 금지 / 답변 생성 금지 (Query Drift 방지)
multi_query_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """당신은 RAG 검색용 Multi-Query 생성 도우미입니다.

사용자의 질문과 동일한 정보 요구를 유지하면서,
서로 다른 표현과 관점의 검색 질문 3개를 생성하세요.

규칙:
1. 원본 질문의 핵심 의도를 유지하세요.
2. 새로운 주제나 원본에 없는 세부 조건을 추가하지 마세요.
3. 세 질문은 단순한 어미 변경이 아니라 검색 표현이 의미 있게 달라야 합니다.
4. 질문마다 하나의 명확한 검색 의도만 포함하세요.
5. 답변을 생성하지 마세요.
6. 검색 질문만 한 줄에 하나씩 출력하세요.
7. 번호나 설명은 붙이지 마세요."""
    ),
    ("human", "원본 질문: {question}")
])

# 프롬프트 → LLM → 문자열 변환을 하나의 체인으로 연결 (LCEL)
multi_query_chain = multi_query_prompt | llm | StrOutputParser()


def parse_queries(text: str, max_queries: int = 3) -> list[str]:
    """LLM 출력에서 검색 질문만 추출하고 중복을 제거한다."""
    queries = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        # 모델이 번호/불릿을 붙인 경우 제거 (예: "1. ", "- ", "• ")
        line = re.sub(r"^[-*•\s]*\d*[.)]?\s*", "", line).strip()

        # 같은 질문이 두 번 생성되면 한 번만 사용
        if line and line not in queries:
            queries.append(line)

    return queries[:max_queries]


In [5]:
# LLM으로 검색 질문 3개 생성 후 파싱
generated_text = multi_query_chain.invoke({"question": question})
generated_queries = parse_queries(generated_text)

print("[원본 질문]")
print(question)

print("\n[생성된 검색 질문]")
for i, q in enumerate(generated_queries, start=1):
    print(f"{i}. {q}")


[원본 질문]
2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?

[생성된 검색 질문]
1. 년 AI 기술 발전 전망은 어떤 방향으로 예상되나요?
2. 년에 인공지능 기술은 어떤 분야로 진화할 것으로 보이나요?
3. 년 AI 기술의 향후 발전 추세는 어떻게 전망되나요?


## 생성된 질문을 먼저 검토한다

검색하기 전에 다음을 확인한다.

```text
[ ] 세 질문 모두 원본 질문과 같은 정보를 찾고 있는가?
[ ] 서로 다른 표현을 사용하고 있는가?
[ ] 원본 질문에 없던 새로운 조건을 만들지 않았는가?
[ ] 답변이나 결론을 미리 넣지 않았는가?
```

Multi Query도 LLM이 생성하므로 **Query Drift**가 발생할 수 있다.

즉, 질문을 다양화하다가 원래 사용자의 의도에서 벗어날 수 있다.


# 5. 여러 Query로 각각 검색하기

이번에는 다음처럼 검색한다.

```text
원본 질문
   │
   ├─ Original Query ──→ Top-k
   │
   ├─ Query 1 ─────────→ Top-k
   │
   ├─ Query 2 ─────────→ Top-k
   │
   └─ Query 3 ─────────→ Top-k
```

원본 질문도 함께 검색한다.

이유는 LLM이 생성한 질의가 원래 질문의 일부 의미를 놓치는 경우에도
원본 질문 검색 결과를 유지하기 위해서이다.


In [6]:
def retrieve_each_query(queries: list[str], k: int = 3):
    """각 query별 Dense Retrieval 결과를 반환한다."""
    results_by_query = []

    # query마다 독립적으로 Top-k 검색 (아직 병합하지 않음)
    for query in queries:
        results = vector_store.similarity_search_with_score(query, k=k)

        results_by_query.append({
            "query": query,
            "results": results,
        })

    return results_by_query


In [7]:
# 원본 질문 + 생성된 질문을 모두 검색 대상으로 사용
# 원본을 포함하는 이유: 생성된 질문이 원본의 일부 의미를 놓쳐도 원본 검색 결과를 유지하기 위해
search_queries = [question] + generated_queries

results_by_query = retrieve_each_query(
    queries=search_queries,
    k=3,
)

# query별로 어떤 chunk가 검색되었는지 확인 (병합 전 결과)
for query_result in results_by_query:
    print("\n" + "=" * 110)
    print(f"[Query] {query_result['query']}")
    print("=" * 110)

    for rank, (doc, score) in enumerate(query_result["results"], start=1):
        print(
            f"{rank}. distance={score:.4f}"
            f" | chunk_id={doc.metadata.get('chunk_id')}"
            f" | pages={doc.metadata.get('pages')}"
            f" | heading={doc.metadata.get('heading')}"
        )



[Query] 2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
1. distance=0.3202 | chunk_id=10 | pages=5 | heading=AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
2. distance=0.3710 | chunk_id=38 | pages=17 | heading=4.2 2026년 AI 산업 전망
3. distance=0.3890 | chunk_id=2 | pages=1 | heading=토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 > 목차

[Query] 년 AI 기술 발전 전망은 어떤 방향으로 예상되나요?
1. distance=0.3339 | chunk_id=10 | pages=5 | heading=AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
2. distance=0.3379 | chunk_id=38 | pages=17 | heading=4.2 2026년 AI 산업 전망
3. distance=0.3789 | chunk_id=40 | pages=18 | heading=4.3 2026년 AI 기술 전망

[Query] 년에 인공지능 기술은 어떤 분야로 진화할 것으로 보이나요?
1. distance=0.4389 | chunk_id=10 | pages=5 | heading=AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
2. distance=0.4920 | chunk_id=11 | pages=6 | heading=1.1 분석 배경
3. distance=0.4967 | chunk_id=38 | pages=17 | heading=4.2 2026년 AI 산업 전망

[Query] 년 AI 기술의 향후 발전 추세는 어떻게 전망되나요?
1. distance=0.3440 | chunk_id=10 | pages=5 | heading=AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
2. distance=0.3630 | chun

# 6. 검색 결과 합치기와 중복 제거

여러 질의로 검색하면 같은 chunk가 반복해서 검색될 수 있다.

예:

```text
Original → C10, C21, C35
Query 1  → C21, C18, C35
Query 2  → C35, C42, C10
Query 3  → C18, C51, C21
```

단순히 합치면 12개 문서처럼 보이지만 실제 unique chunk는 더 적다.

따라서 Multi Query의 기본 처리 과정은 다음과 같다.

```text
각 Query 검색
     ↓
검색 결과 합치기
     ↓
중복 chunk 제거
     ↓
최종 후보 문서
```

이번 단계에서는 **RRF나 점수 융합을 하지 않는다.**

순위 융합은 다음 Retrieval Engineering 실습에서 별도로 학습한다.


In [8]:
def document_key(doc) -> str:
    """중복 제거를 위한 문서 식별키를 만든다."""
    chunk_id = doc.metadata.get("chunk_id")

    # chunk_id는 문서 내용과 무관하게 chunk마다 고유하므로 가장 신뢰할 수 있는 키
    if chunk_id is not None:
        return f"chunk:{chunk_id}"

    # chunk_id가 없는 경우를 위한 fallback (본문 해시로 식별)
    return f"text:{hash(doc.page_content)}"


def merge_unique_documents(results_by_query):
    """query 순서와 검색 순서를 유지하며 중복 문서를 제거한다."""
    unique_docs = []
    seen = set()  # 이미 등장한 문서 키를 기록

    # 앞쪽 query의 결과가 먼저 들어오므로, 순서가 곧 우선순위가 된다
    for query_result in results_by_query:
        for doc, score in query_result["results"]:
            key = document_key(doc)

            # 이미 추가된 chunk는 건너뛴다 (중복 제거)
            if key in seen:
                continue

            seen.add(key)
            unique_docs.append(doc)

    return unique_docs


In [9]:
multi_docs = merge_unique_documents(results_by_query)

print(f"검색 query 수          : {len(search_queries)}")
print(f"query당 검색 문서 수  : 3")
print(f"최대 후보 수          : {len(search_queries) * 3}")
print(f"중복 제거 후 문서 수  : {len(multi_docs)}")

print("\n[중복 제거 후 문서]")
for i, doc in enumerate(multi_docs, start=1):
    print(
        f"{i}. chunk_id={doc.metadata.get('chunk_id')}"
        f" | pages={doc.metadata.get('pages')}"
        f" | heading={doc.metadata.get('heading')}"
    )


검색 query 수          : 4
query당 검색 문서 수  : 3
최대 후보 수          : 12
중복 제거 후 문서 수  : 5

[중복 제거 후 문서]
1. chunk_id=10 | pages=5 | heading=AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
2. chunk_id=38 | pages=17 | heading=4.2 2026년 AI 산업 전망
3. chunk_id=2 | pages=1 | heading=토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 > 목차
4. chunk_id=40 | pages=18 | heading=4.3 2026년 AI 기술 전망
5. chunk_id=11 | pages=6 | heading=1.1 분석 배경


# 7. Single Query vs Multi Query 비교

Multi Query의 핵심은 단순히 **문서 수를 늘리는 것**이 아니다.

다음 관점으로 비교한다.

1. Single Query에서 찾지 못한 관련 chunk가 추가되었는가?
2. 서로 다른 page / section의 관련 근거가 추가되었는가?
3. 불필요한 문서도 함께 증가하지 않았는가?
4. 원래 질문과 관련성이 낮은 문서가 생기지는 않았는가?

> **검색 결과가 많아졌다는 사실만으로 성능이 향상되었다고 결론 내리면 안 된다.**


In [10]:
def get_doc_ids(docs):
    # chunk_id 집합을 만들어 Single / Multi 결과를 집합 연산으로 비교하기 쉽게 한다
    return {
        doc.metadata.get("chunk_id")
        for doc in docs
        if doc.metadata.get("chunk_id") is not None
    }


single_docs = [doc for doc, _ in single_results]  # (doc, score) 튜플에서 doc만 추출

single_ids = get_doc_ids(single_docs)
multi_ids = get_doc_ids(multi_docs)

# Multi Query에서 새로 발견된 chunk = 차집합 (multi - single)
added_ids = multi_ids - single_ids

single_pages = {
    doc.metadata.get("pages")
    for doc in single_docs
}

multi_pages = {
    doc.metadata.get("pages")
    for doc in multi_docs
}

print("[Single Query]")
print(f"unique chunk 수 : {len(single_ids)}")
print(f"검색 page 범위  : {sorted(single_pages)}")

print("\n[Multi Query]")
print(f"unique chunk 수 : {len(multi_ids)}")
print(f"검색 page 범위  : {sorted(multi_pages)}")

print("\n[Multi Query에서 새롭게 추가된 chunk_id]")
print(sorted(added_ids))


[Single Query]
unique chunk 수 : 4
검색 page 범위  : ['1', '17', '18', '5']

[Multi Query]
unique chunk 수 : 5
검색 page 범위  : ['1', '17', '18', '5', '6']

[Multi Query에서 새롭게 추가된 chunk_id]
[11]


## 비교 결과를 해석하는 방법

좋은 결과:

```text
Single Query
→ 핵심 section 일부 검색

Multi Query
→ 같은 주제의 다른 관련 section까지 추가 검색
```

좋지 않은 결과:

```text
Single Query
→ 관련 문서 4개

Multi Query
→ 관련 문서 4개 + 주변 주제 문서 7개
```

따라서 Multi Query는 일반적으로 **Recall을 높이는 방향**으로 작동하지만,
동시에 noise가 늘어날 수 있다.

이 문제를 뒤에서 다음 기술로 보완할 수 있다.

```text
Multi Query
      ↓
넓은 후보 문서
      ↓
Reranking
      ↓
관련성이 높은 문서만 선택
```


# 8. Multi-Query RAG 만들기

검색 단계에서는 여러 query를 사용하지만,
최종 답변은 **사용자의 원본 질문**에 답해야 한다.

```text
원본 질문
     ↓
Multi Query 생성
     ↓
여러 Query 검색
     ↓
문서 병합 / 중복 제거
     ↓
Context
     ↓
원본 질문 + Context
     ↓
LLM
     ↓
최종 답변
```

Context가 지나치게 커지지 않도록 이번 예제에서는
중복 제거 후 앞쪽의 최대 6개 문서만 사용한다.

다음 실습에서 RRF, 이후 Reranking을 배우면
이 문서 선택 과정을 더 정교하게 개선할 수 있다.


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# 검색된 문서들을 LLM에 넣을 하나의 context 문자열로 합친다
def format_docs(docs):
    parts = []

    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata

        # 각 문서에 페이지/섹션 정보를 붙여 LLM이 출처를 구분할 수 있게 한다
        parts.append(
            f"[문서 {i} | pages={meta.get('pages')} | heading={meta.get('heading')}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(parts)


In [12]:

# 답변 생성 프롬프트: 검색 문서에 근거해서만 답하도록 제한 (환각 방지)
answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """아래 검색 문서에 근거하여 사용자의 질문에 답하세요.

규칙:
1. 검색 문서에서 확인되는 내용만 사용하세요.
2. 문서에서 확인되지 않는 내용은 추측하지 마세요.
3. 핵심 내용을 이해하기 쉽게 설명하세요.

[검색 문서]
{context}"""
    ),
    ("human", "{question}")
])

In [13]:
def multi_query_rag(
    question: str,
    num_queries: int = 3,
    per_query_k: int = 3,
    max_context_docs: int = 6,
):
    # 1. 검색용 query 여러 개 생성
    generated_text = multi_query_chain.invoke({
        "question": question
    })

    generated_queries = parse_queries(
        generated_text,
        max_queries=num_queries,
    )

    # 2. 원본 질문까지 포함하여 검색
    search_queries = [question] + generated_queries

    results_by_query = retrieve_each_query(
        queries=search_queries,
        k=per_query_k,
    )

    # 3. 검색 결과 병합 + 중복 제거
    docs = merge_unique_documents(results_by_query)

    # 4. context 크기 제한 (앞쪽 문서 우선 사용)
    context_docs = docs[:max_context_docs]
    context = format_docs(context_docs)

    # 5. 최종 답변은 원본 질문으로 생성 (생성된 query가 아니라)
    messages = answer_prompt.invoke({
        "question": question,
        "context": context,
    })

    response = llm.invoke(messages)
    answer = StrOutputParser().invoke(response)

    # 중간 결과를 함께 반환하여 각 단계를 확인할 수 있게 한다
    return {
        "original_question": question,
        "generated_queries": generated_queries,
        "search_queries": search_queries,
        "documents": docs,
        "context_documents": context_docs,
        "answer": answer,
    }


In [14]:
result = multi_query_rag(
    "2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?",
    num_queries=3,
    per_query_k=3,
    max_context_docs=6,
)

print("[원본 질문]")
print(result["original_question"])

print("\n[생성된 Multi Queries]")
for i, q in enumerate(result["generated_queries"], start=1):
    print(f"{i}. {q}")

# 중복 제거 후 남은 전체 후보 수 (LLM context에 들어간 수와 다를 수 있음)
print("\n[검색된 unique chunk 수]")
print(len(result["documents"]))

print("\n[최종 Context에 사용한 문서]")
for i, doc in enumerate(result["context_documents"], start=1):
    print(
        f"{i}. pages={doc.metadata.get('pages')}"
        f" | heading={doc.metadata.get('heading')}"
    )

print("\n[최종 답변]")
print(result["answer"])


[원본 질문]
2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?

[생성된 Multi Queries]
1. 년 AI 기술 발전은 어떤 방향으로 예상되나요?
2. 년에 인공지능 기술은 어떤 추세로 발전할까요?
3. 년 AI 산업의 기술적 발전 전망은 어떻게 되나요?

[검색된 unique chunk 수]
6

[최종 Context에 사용한 문서]
1. pages=5 | heading=AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
2. pages=17 | heading=4.2 2026년 AI 산업 전망
3. pages=1 | heading=토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 > 목차
4. pages=18 | heading=4.3 2026년 AI 기술 전망
5. pages=6 | heading=1.1 분석 배경
6. pages=18 | heading=4.2 2026년 AI 산업 전망

[최종 답변]
2026년 AI 기술은 **지능 구조 고도화와 데이터 한계 보완**을 중심으로 발전할 것으로 전망됩니다.

핵심 방향은 다음과 같습니다.

- **합성데이터, 추론형 AI, 멀티모달 기술**이 주요 경쟁 축으로 자리 잡을 가능성이 큼
- 이를 통해 **학습 효율 향상**, **복합 정보 처리**, **설명 가능성 강화** 같은 모델 내부 구조의 질적 개선이 이어질 것으로 예상됨
- 입력 유형과 데이터 범위가 넓어지면서, AI의 **활용 가능성**과 **복잡한 상황에서의 판단 능력**도 강화되는 방향으로 진화할 전망임
- **고품질 데이터 생성 + 멀티모달 + 고급 추론**이 결합되며, AI의 **상황 이해와 문제 해결 능력**이 향상되고 서비스·산업 전반의 활용도도 확대될 것으로 보임
- 기술이 고도화될수록 **안전성·정합성 검증**의 중요성도 커져, **신뢰성 평가와 표준화**가 확산의 기반 인프라로 자리 잡을 것으로 전망됨
- 특히 **데이터 품질 표준화**와 **모델 신뢰성 평가 기준 구축**이 산업 적용 확산

# 9. Multi Query의 장점과 한계

## 장점

### 1. 표현 차이 보완

```text
사용자 표현
↔
문서 표현
```

두 표현이 다르더라도 여러 query를 사용하면 관련 문서를 찾을 가능성이 높아진다.

### 2. Recall 확대

하나의 query에서 놓친 관련 chunk를 다른 query가 찾을 수 있다.

### 3. 모호한 질문 보완

질문의 의미를 여러 표현으로 탐색할 수 있다.

---

## 한계

### 1. 검색 비용 증가

```text
Single Query
1회 검색

Multi Query
원본 + 생성 Query 수만큼 검색
```

### 2. Latency 증가

LLM으로 query를 생성하는 호출도 추가된다.

### 3. Noise 증가

후보 문서가 많아지는 만큼 관련성이 낮은 문서도 들어올 수 있다.

### 4. Query Drift

LLM이 원본 질문의 의미를 잘못 확장할 수 있다.

---

따라서 Multi Query의 핵심은 다음과 같다.

> **더 많이 검색하는 기술이 아니라, 같은 정보 요구를 여러 표현으로 검색하여 관련 문서를 놓칠 가능성을 줄이는 기술이다.**


# 10. 미니 실습

다음 질문 중 하나를 선택한다.

```text
A. AI 규제는 2026년에 어떤 방향으로 바뀔 것으로 전망되나요?

B. 기업 내부에서 AI 에이전트는 어떤 방식으로 활용될 것으로 전망되나요?

C. 추론형 데이터는 어떤 유형으로 구분되나요?
```

## 수행 순서

1. 원본 질문으로 Top-3 검색
2. Multi Query 3개 생성
3. 생성된 질문이 원본 의도를 유지하는지 검토
4. 원본 + 3개 Query로 각각 Top-3 검색
5. 중복 제거
6. Single Query와 Multi Query의 unique chunk 비교
7. 추가된 chunk가 실제로 관련 있는지 판단

## 학생이 설명해야 할 것

```text
1. 왜 이 질문에서 Multi Query가 필요한가?
2. 생성된 query 중 가장 유용한 것은 무엇인가?
3. Single Query에서 놓친 관련 근거가 있었는가?
4. Multi Query 때문에 불필요한 문서도 늘어났는가?
```


# 11. 03번과 04번의 차이 정리

```text
03 Query Rewriting

"AI 규제 앞으로 어떻게 돼요?"
        ↓
"2026년 AI 정책 환경과 규제 체계는
어떤 방향으로 변화할 것으로 전망되는가?"
        ↓
검색 1회
```

```text
04 Multi Query

"AI 규제 앞으로 어떻게 돼요?"
        ↓
Q1. 2026년 AI 정책 및 규제 변화 방향은?
Q2. AI 안전·책임 중심 규제는 어떻게 변화하는가?
Q3. 글로벌 AI 규제와 국내 제도는 어떤 방향으로 정비되는가?
        ↓
검색 여러 회
        ↓
문서 병합
```

### 기억해야 할 것

```text
Query Rewriting
→ Precision을 높이기 위한 질의 명확화에 가깝다.

Multi Query
→ Recall을 높이기 위한 검색 범위 확장에 가깝다.
```

정확히는 두 기법 모두 상황에 따라 precision/recall에 영향을 줄 수 있지만,
초보 단계에서는 위 관점으로 이해하면 전체 구조를 잡기 쉽다.


# 12. 정리 및 다음 실습

현재까지 Advanced RAG 흐름은 다음과 같다.

```text
01 Naive RAG
        ↓
02 Document Processing
   Docling
   Structure-aware Chunking
   Metadata
        ↓
03 Query Rewriting
   1 Question → 1 Better Query
        ↓
04 Multi Query
   1 Question → Multiple Queries
        ↓
05 Retrieval Engineering
   Dense
   BM25
   Hybrid
   RRF
```

## 이번 실습 핵심

1. 하나의 질문을 서로 다른 검색 표현으로 확장했다.
2. 원본 질문도 검색에 포함했다.
3. 각 query의 검색 결과를 하나로 모았다.
4. 같은 chunk를 중복 제거했다.
5. Single Query와 Multi Query의 검색 범위를 비교했다.
6. Multi Query는 Recall 확대에 유리하지만 비용·latency·noise가 증가할 수 있다.

## 다음 질문

이제 새로운 문제가 생긴다.

```text
Dense Retrieval만 사용하는 것이 항상 최선인가?
```

정확한 용어, 고유명사, 정책명처럼 **키워드 일치가 중요한 질문**에서는
BM25 같은 Sparse Retrieval이 강할 수 있다.

따라서 다음 실습에서는

```text
Dense
   +
BM25
   ↓
Hybrid Retrieval
   ↓
RRF
```

순서로 Retrieval Engineering을 학습한다.
